# Live points price

Odds come from the newest JSON file in each NBA book folder under `data/props`. Models and feature rows stay on disk. This notebook does not fetch a lineup.

1. Read the latest points prices from those files.
2. Turn each over/under pair into a no-vig probability.
3. Run the minutes feature row through the minutes bundle and the rate feature row through the rate bundle. Each returns 11 quantiles.
4. Draw a minutes uniform and a rate uniform independently, multiply, and count the integers into a PMF.
5. Read `P(over the line)` from that PMF and set it next to the book's probability.


In [23]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("could not find repo root (pyproject.toml)")

ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.shared.artifacts import load_model_bundle, monotonize_quantiles, predict_quantiles
from models.shared.minutes_sampler import load_tail_sidecar as load_minutes_tails
from models.shared.ppm_sampler import load_tail_sidecar as load_rate_tails
from models.shared.simulate import line_probs, points_pmf
from src.models.odds import remove_vig

N_DRAWS = 2_000
SAVED = ROOT / "models/saved_models"


## Latest points prices

Each book uses the newest file in its NBA folder. Novig, ProphetX, and Pinnacle use the props file, not the team file. Only the main full-game points line is kept for each player. FanDuel's line is the `Player - Points` market. DraftKings does not flag a main line, so the ladder line whose price is closest to even money is kept. Novig and ProphetX use the file's `is_main` flag. Quarter, milestone, and the rest of the alternate ladder are dropped.


In [24]:
BOOKS = (
    ("fanduel", ROOT / "data/props/fanduel/nba", "picks"),
    ("draftkings", ROOT / "data/props/draftkings/nba", "picks"),
    ("pinnacle", ROOT / "data/props/pinnacle/nba", "games"),
    ("novig", ROOT / "data/props/novig/nba", "games"),
    ("prophetx", ROOT / "data/props/prophetx/nba", "games"),
    ("underdog", ROOT / "data/props/underdogs/nba", "picks"),
    ("prizepicks", ROOT / "data/props/prizepicks/nba", "projections"),
)


def _latest_file(folder: Path, kind: str) -> Path:
    files = [path for path in folder.glob("*.json") if path.is_file()]
    if kind == "games":
        props = [path for path in files if path.name.endswith("_props.json")]
        files = props or files
    if not files:
        raise FileNotFoundError(f"no json files in {folder}")
    return max(files, key=lambda path: path.stat().st_mtime)


def _prop_name(value) -> str:
    return str(value or "").strip().lower()


def _is_points(value) -> bool:
    return _prop_name(value) == "points"


def _pick_is_main(market_name: object) -> bool | None:
    """FanDuel's full-game market is already one line. DraftKings' ladder is chosen later."""
    market = str(market_name or "").strip()
    if not market:
        return True
    if market.endswith(" - Points"):
        return True
    lowered = market.lower()
    if lowered.startswith("points - ") and lowered.endswith(" points") and "1st" not in lowered:
        return None
    return False


def _event_fields(record: dict, game: dict | None = None) -> dict:
    source = game or {}
    return {
        "event_name": record.get("event_name") or source.get("name") or source.get("label"),
        "start_time": record.get("start_time") or source.get("start_time") or source.get("scheduled"),
    }


def _points_rows(payload: dict, kind: str) -> list[dict]:
    fetched = payload.get("fetched_at")
    rows = []
    if kind == "picks":
        for pick in payload.get("picks") or []:
            if not _is_points(pick.get("stat_name")):
                continue
            rows.append({
                "player_name": pick.get("full_name"),
                "prop": _prop_name(pick.get("stat_name")),
                "side": pick.get("choice"),
                "line_score": pick.get("stat_value"),
                "american_price": pick.get("american_price"),
                "is_main": _pick_is_main(pick.get("market_name")),
                "fetched_at": fetched,
                **_event_fields(pick),
            })
        return rows
    if kind == "projections":
        for projection in payload.get("projections") or []:
            if not _is_points(projection.get("stat_type")):
                continue
            if str(projection.get("odds_type") or "").strip().lower() != "standard":
                continue
            rows.append({
                "player_name": projection.get("player"),
                "prop": _prop_name(projection.get("stat_type")),
                "line_score": projection.get("line_score"),
                "is_main": True,
                "fetched_at": fetched,
                "event_name": None,
                "start_time": None,
            })
        return rows
    for game in payload.get("games") or []:
        for prop in game.get("props") or []:
            if not _is_points(prop.get("stat")):
                continue
            event = _event_fields(prop, game)
            is_main = prop.get("is_main", True)
            if "american_over" in prop or "american_under" in prop:
                for side, price_key in (("over", "american_over"), ("under", "american_under")):
                    rows.append({
                        "player_name": prop.get("player"),
                        "prop": _prop_name(prop.get("stat")),
                        "side": side,
                        "line_score": prop.get("line"),
                        "american_price": prop.get(price_key),
                        "is_main": is_main,
                        "fetched_at": fetched,
                        **event,
                    })
                continue
            for side in ("over", "under"):
                price = prop.get(side) or {}
                if not isinstance(price, dict):
                    continue
                rows.append({
                    "player_name": prop.get("player"),
                    "prop": _prop_name(prop.get("stat")),
                    "side": side,
                    "line_score": prop.get("line"),
                    "american_price": price.get("american"),
                    "is_main": is_main,
                    "fetched_at": fetched,
                    **event,
                })
    return rows


def _name_key(names: pd.Series) -> pd.Series:
    return (
        names.astype("string")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def _as_quotes(raw: pd.DataFrame, book: str) -> pd.DataFrame:
    columns = [
        "book", "player", "prop", "name_key", "line",
        "over_american", "under_american", "fetched_at",
        "event_name", "start_time", "is_main",
    ]
    if raw.empty:
        return pd.DataFrame(columns=columns)
    frame = raw.copy()
    frame["player"] = frame["player_name"].astype(str)
    frame["prop"] = frame["prop"].map(_prop_name)
    frame["line"] = pd.to_numeric(frame["line_score"], errors="coerce")
    frame["fetched_at"] = frame["fetched_at"]
    if "is_main" not in frame.columns:
        frame["is_main"] = True
    for column in ("event_name", "start_time"):
        if column not in frame.columns:
            frame[column] = pd.NA
    meta = ["fetched_at", "event_name", "start_time", "is_main"]
    if "side" not in frame.columns:
        out = frame.loc[:, ["player", "prop", "line", *meta]].drop_duplicates(["player", "prop", "line"])
        out["book"] = book
        out["over_american"] = np.nan
        out["under_american"] = np.nan
    else:
        frame["side"] = frame["side"].astype(str).str.lower()
        frame["american_price"] = pd.to_numeric(frame["american_price"], errors="coerce")
        keys = ["player", "prop", "line"]
        base = frame.drop_duplicates(keys, keep="last")[keys + meta]
        over = (
            frame.loc[frame["side"].eq("over"), keys + ["american_price"]]
            .drop_duplicates(keys, keep="last")
            .rename(columns={"american_price": "over_american"})
        )
        under = (
            frame.loc[frame["side"].eq("under"), keys + ["american_price"]]
            .drop_duplicates(keys, keep="last")
            .rename(columns={"american_price": "under_american"})
        )
        out = base.merge(over, on=keys, how="left").merge(under, on=keys, how="left")
        out["book"] = book
    out["name_key"] = _name_key(out["player"])
    return out.loc[:, columns].reset_index(drop=True)


frames = []
for book, folder, kind in BOOKS:
    path = _latest_file(folder, kind)
    payload = json.loads(path.read_text())
    raw = pd.DataFrame(_points_rows(payload, kind))
    fetched = payload.get("fetched_at")
    print(f"{book:12} rows={len(raw):5}  file={path.name}  fetched_at={fetched}")
    frames.append(_as_quotes(raw, book))

quotes = pd.concat(frames, ignore_index=True)
if quotes.empty:
    raise ValueError("no player-points rows in the latest NBA odds files")

from src.models.odds import american_to_implied_probability

def _distance_from_even(over, under) -> float:
    def dist(odds) -> float:
        if pd.isna(odds):
            return 1.0
        price = float(odds)
        if price == 0:
            return 1.0
        return abs(american_to_implied_probability(price) - 0.5)
    return dist(over) + dist(under)

quotes["is_main"] = quotes["is_main"].astype("boolean")
pending = quotes.index[quotes["is_main"].isna()]
if len(pending):
    choice = quotes.loc[pending, ["book", "name_key"]].copy()
    choice["_score"] = [
        _distance_from_even(over, under)
        for over, under in zip(
            quotes.loc[pending, "over_american"],
            quotes.loc[pending, "under_american"],
        )
    ]
    winners = (
        choice.sort_values("_score", kind="mergesort")
        .groupby(["book", "name_key"], sort=False)
        .head(1)
        .index
    )
    quotes.loc[pending, "is_main"] = False
    quotes.loc[winners, "is_main"] = True

listed = len(quotes)
quotes = quotes.loc[quotes["is_main"].eq(True)].drop(columns=["is_main"]).reset_index(drop=True)
print(
    f"main lines {len(quotes):,}",
    f"dropped alts {listed - len(quotes):,}",
    f"players {quotes['player'].nunique():,}",
)


fanduel      rows=  190  file=fanduel_nba_2026-10-23_192539.json  fetched_at=2026-10-24T02:25:39Z
draftkings   rows=    0  file=draftkings_nba_2026-10-23_192536.json  fetched_at=2026-10-24T02:25:36Z
pinnacle     rows=    0  file=pinnacle_nba_2026-10-23_192536_props.json  fetched_at=2026-10-23T19:25:44-07:00
novig        rows=  610  file=novig_nba_2026-10-23_192537_props.json  fetched_at=2026-10-23T19:25:37-07:00
prophetx     rows=  274  file=prophetx_nba_2026-10-23_192537_props.json  fetched_at=2026-10-23T19:25:37-07:00
underdog     rows=   24  file=underdog_nba_2026-10-23_192543.json  fetched_at=2026-10-24T02:25:43Z
prizepicks   rows=   12  file=prizepicks_nba_2026-10-23_192536.json  fetched_at=2026-10-24T02:25:36Z
main lines 205 dropped alts 390 players 103


## Book probability

A posted price already includes the vig. The table is one row per side: `odds`, the raw `implied_prob`, and `no_vig_prob` after both sides are scaled to sum to 1. `quotes["book_over"]` keeps the over side's no-vig probability for the edge below. PrizePicks has a line and no price, so those rows stay out of the table.


In [25]:
from src.models.odds import american_to_implied_probability


def _side_prices(over_american, under_american):
    blank = (np.nan, np.nan, np.nan, np.nan)
    if pd.isna(over_american) or pd.isna(under_american):
        return blank
    try:
        over_odds = float(over_american)
        under_odds = float(under_american)
        over_implied = american_to_implied_probability(over_odds)
        under_implied = american_to_implied_probability(under_odds)
        no_vig_over, no_vig_under = remove_vig(over_odds, under_odds)
    except ValueError:
        return blank
    return over_implied, under_implied, no_vig_over, no_vig_under


prices = [
    _side_prices(over, under)
    for over, under in zip(quotes["over_american"], quotes["under_american"])
]
quotes["book_over"] = [no_vig_over for _, _, no_vig_over, _ in prices]

rows = []
quote_fields = quotes[["book", "player", "prop", "line", "over_american", "under_american"]]
for (book, player, prop, line, over_odds, under_odds), price in zip(
    quote_fields.itertuples(index=False, name=None),
    prices,
):
    over_implied, under_implied, no_vig_over, no_vig_under = price
    if pd.isna(no_vig_over):
        continue
    rows.append({
        "book": book,
        "player": player,
        "prop": prop,
        "line": line,
        "side": "over",
        "odds": over_odds,
        "implied_prob": over_implied,
        "no_vig_prob": no_vig_over,
    })
    rows.append({
        "book": book,
        "player": player,
        "prop": prop,
        "line": line,
        "side": "under",
        "odds": under_odds,
        "implied_prob": under_implied,
        "no_vig_prob": no_vig_under,
    })

print(quotes.groupby("book").size().rename("lines").to_string())
pd.DataFrame(rows).head(12)


book
fanduel       61
novig         87
prizepicks    12
prophetx      33
underdog      12


,book,player,prop,line,side,odds,implied_prob,no_vig_prob
0,fanduel,Dejounte Murray,points,11.5,over,-112.0,0.528302,0.493931
1,fanduel,Dejounte Murray,points,11.5,under,-118.0,0.541284,0.506069
2,fanduel,Darryn Peterson,points,13.5,over,-106.0,0.514563,0.480847
3,fanduel,Darryn Peterson,points,13.5,under,-125.0,0.555556,0.519153
4,fanduel,Trey Murphy III,points,15.5,over,-118.0,0.541284,0.506069
5,fanduel,Trey Murphy III,points,15.5,under,-112.0,0.528302,0.493931
6,fanduel,Jaxson Hayes,points,11.5,over,104.0,0.490196,0.458115
7,fanduel,Jaxson Hayes,points,11.5,under,-138.0,0.579832,0.541885
8,fanduel,Lauri Markkanen,points,24.5,over,-110.0,0.523810,0.489879
9,fanduel,Lauri Markkanen,points,24.5,under,-120.0,0.545455,0.510121


## Starting five

`projected_lineups` fetches the RotoWire page and returns `{team abbreviation: [five names]}`. The price cell reads `lineups` from here. A team missing from the page has no starters, so those players stay at `starting` 0.

In [26]:
from src.live.nba_lineup import projected_lineups

lineups = projected_lineups()
print(f"{len(lineups)} teams")
pd.DataFrame(
    [
        {"team": team, "starters": ", ".join(names)}
        for team, names in sorted(lineups.items())
    ]
)

24 teams


,team,starters
0,ATL,"CJ McCollum, Nickeil Alexander-Walker, Dyson D..."
1,BOS,"Derrick White, Paul George, Jayson Tatum, Jord..."
2,CHA,"Dennis Schroder, Sion James, Brandon Miller, N..."
3,CHI,"Josh Giddey, Norman Powell, Matas Buzelis, Cal..."
4,DAL,"Kyrie Irving, Max Christie, P.J. Washington, D..."
5,DET,"Cade Cunningham, Duncan Robinson, Ausar Thomps..."
6,GSW,"Stephen Curry, Brandin Podziemski, Yaxel Lende..."
7,HOU,"Fred VanVleet, Amen Thompson, Kevin Durant, Ja..."
8,IND,"Tyrese Haliburton, Andrew Nembhard, Jarace Wal..."
9,LAC,"Darius Garland, Kris Dunn, Rui Hachimura, Derr..."


## Model price

The cell below builds `minutes_live` and `rate_live` from silver gamelogs and the quoted players. Re-run the odds cell first so each quote carries `event_name`. Run the starting-five cell first. `lineups` is what marks `starting`. A player is included when their last silver team is one of the two teams on an odds event. Underdog and PrizePicks rows have no event, so those players take the one game their last team has in the other books. History is every silver game before the earliest quoted tip. These files have no spread, so `team_spread_canonical` is missing on tonight's rows.

`price_slate` predicts both quantile sets, draws minutes and rate independently, and adds `model_over` for every quoted line. `edge` is `model_over` minus `book_over`. `ev` is the expected profit per $1 staked on the over at the posted `over_american` price, juice included. A push returns the stake, so the loss probability is the model's under, not `1 - model_over`. A quote with no American price has no `ev`. `N_DRAWS` is 2,000 here so the first pass finishes. The priced path uses 10,000. The table keeps lines whose tip is today in Pacific time. A quote with no start time stays out of that table.


In [27]:
from src.live.minutes_live import build_minutes_live
from src.live.ppm_live import build_ppm_live
from src.models.odds import expected_value
from src.pipeline.silver.positions import player_name_keys
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.nba.nba_lineups import NBA_TEAM_ABBREVIATIONS

minutes_bundle = load_model_bundle(SAVED / "min_nba_model_2026-04-12.joblib")
rate_bundle = load_model_bundle(SAVED / "pts_nba_model_2026-04-12.joblib")
minutes_tails = load_minutes_tails(SAVED / "min_nba_tails_2026-04-12.joblib")
rate_tails = load_rate_tails(SAVED / "pts_nba_tails_2026-04-12.joblib")
print("minutes features:", ", ".join(minutes_bundle["feature_names"]))
print("rate features:", ", ".join(rate_bundle["feature_names"]))


def _quantiles(bundle, frame, prefix: str) -> pd.DataFrame:
    names = list(bundle["feature_names"])
    missing = [name for name in names if name not in frame.columns]
    if missing:
        raise KeyError(f"{prefix} features missing {missing}")
    predicted = predict_quantiles(
        bundle["quantile_models"],
        names,
        frame.loc[:, names],
    )
    grid = monotonize_quantiles(
        {column: predicted[column].to_numpy() for column in predicted.columns}
    )
    out = pd.DataFrame(index=frame.index)
    for key, values in grid.items():
        level = str(key).split("_", 1)[1]
        out[f"{prefix}_q_{level}"] = values
    return out


def price_slate(
    minutes_live: pd.DataFrame,
    rate_live: pd.DataFrame,
    quotes: pd.DataFrame,
    n_draws: int = N_DRAWS,
) -> pd.DataFrame:
    keys = ["player_id", "game_id"]
    for label, frame, extra in (
        ("minutes", minutes_live, ["player_name", "starting"]),
        ("rate", rate_live, []),
    ):
        missing = [column for column in [*keys, *extra] if column not in frame.columns]
        if missing:
            raise KeyError(f"{label} frame missing {missing}")

    minutes_q = _quantiles(minutes_bundle, minutes_live, "minutes")
    rate_q = _quantiles(rate_bundle, rate_live, "rate")
    left = minutes_live.loc[:, [*keys, "player_name", "starting"]].join(minutes_q)
    right = rate_live.loc[:, keys].join(rate_q)
    rows = left.merge(right, on=keys, how="inner").reset_index(drop=True)
    if rows.empty:
        raise ValueError("no player-game is in both feature frames")

    pmf, mean, median = points_pmf(
        rows,
        n_draws=n_draws,
        minutes_tables=minutes_tails,
        rate_tables=rate_tails,
    )
    rows["model_mean"] = mean
    rows["model_median"] = median
    rows["name_key"] = [
        (player_name_keys(name) or (str(name).lower(),))[0]
        for name in rows["player_name"]
    ]
    keep = ~pd.Series(rows["name_key"]).duplicated(keep="first")
    rows = rows.loc[keep.to_numpy()].reset_index(drop=True)
    pmf = pmf[keep.to_numpy()]
    position: dict[str, int] = {}
    for index, name in enumerate(rows["player_name"]):
        for key in player_name_keys(name):
            position.setdefault(key, index)

    model_over = []
    model_under = []
    model_mean = []
    model_median = []
    for name, line in zip(quotes["player"], quotes["line"]):
        index = next(
            (position[key] for key in player_name_keys(name) if key in position),
            None,
        )
        if index is None or pd.isna(line):
            model_over.append(np.nan)
            model_under.append(np.nan)
            model_mean.append(np.nan)
            model_median.append(np.nan)
            continue
        over, _push, under = line_probs(pmf[index], float(line))
        model_over.append(over)
        model_under.append(under)
        model_mean.append(rows.at[index, "model_mean"])
        model_median.append(rows.at[index, "model_median"])

    card = quotes.copy()
    card["model_over"] = model_over
    card["model_mean"] = model_mean
    card["model_median"] = model_median
    card["edge"] = card["model_over"] - card["book_over"]
    # Loss is the under. A whole-number line can push, and a push returns the
    # stake, so it must not be folded into the loss probability.
    card["ev"] = [
        expected_value(float(win), float(loss), float(odds))
        if pd.notna(win) and pd.notna(loss) and pd.notna(odds) and float(odds) != 0
        else np.nan
        for win, loss, odds in zip(model_over, model_under, card["over_american"])
    ]
    return card


def _load_history() -> pd.DataFrame:
    paths = sorted((ROOT / "data/silver/nba").glob("*/regular_season/player_gamelogs.parquet"))
    if not paths:
        raise FileNotFoundError("no silver player gamelogs under data/silver/nba")
    history = pd.concat([pd.read_parquet(path) for path in paths], ignore_index=True)
    history["game_date"] = pd.to_datetime(history["game_date"])
    return history


def _team_abbr(label: object) -> str | None:
    tokens = str(label or "").replace(".", "").split()
    for token in reversed(tokens):
        abbr = NBA_TEAM_ABBREVIATIONS.get(token)
        if abbr:
            return normalize_team_abbreviation(abbr)
    return None


def _event_teams(event: object) -> set[str]:
    text = str(event or "")
    for sep in (" @ ", " at ", " vs "):
        if sep in text:
            left, right = text.split(sep, 1)
            return {abbr for abbr in (_team_abbr(left), _team_abbr(right)) if abbr}
    return set()


def _tip_date(value) -> pd.Timestamp | None:
    stamp = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(stamp):
        return None
    return pd.Timestamp(stamp).tz_convert(None).normalize()


def _season_label(day: pd.Timestamp) -> str:
    start = int(day.year) if int(day.month) >= 8 else int(day.year) - 1
    return f"{start}-{str(start + 1)[-2:]}"


def _latest_players(history: pd.DataFrame) -> dict[str, object]:
    frame = (
        history.loc[:, ["player_id", "player_name", "team_abbreviation", "team_id", "game_date", "game_id"]]
        .dropna(subset=["player_id", "player_name"])
        .sort_values(["game_date", "game_id"], kind="mergesort")
    )
    lookup: dict[str, object] = {}
    for row in frame.itertuples(index=False):
        for key in player_name_keys(row.player_name):
            lookup[key] = row
    return lookup


def _team_id_lookup(history: pd.DataFrame) -> dict[str, object]:
    frame = history.dropna(subset=["team_abbreviation", "team_id"]).sort_values(
        ["game_date", "game_id"], kind="mergesort"
    )
    frame = frame.assign(abbr=frame["team_abbreviation"].map(normalize_team_abbreviation))
    latest = frame.groupby("abbr", sort=False).tail(1)
    return dict(zip(latest["abbr"], latest["team_id"]))


def build_tonight(history: pd.DataFrame, quotes: pd.DataFrame) -> pd.DataFrame:
    if "event_name" not in quotes.columns:
        raise KeyError("quotes have no event_name; re-run the odds cell")
    players = _latest_players(history)
    team_ids = _team_id_lookup(history)
    events: dict[str, list[tuple]] = {}
    team_games: dict[str, dict[tuple, None]] = {}
    for row in quotes.itertuples(index=False):
        if pd.isna(row.event_name):
            continue
        events.setdefault(row.player, []).append((row.event_name, row.start_time, row.fetched_at))
        sides = _event_teams(row.event_name)
        day = _tip_date(row.start_time) or _tip_date(row.fetched_at)
        if len(sides) != 2 or day is None:
            continue
        for side in sides:
            opponent = next(iter(sides - {side}))
            team_games.setdefault(side, {})[(day, opponent)] = None

    rows = []
    seen: set = set()
    skipped = {"unmatched_name": 0, "no_game": 0, "ambiguous_game": 0, "no_opponent_id": 0}
    for player in quotes["player"].drop_duplicates():
        ident = next((players[key] for key in player_name_keys(player) if key in players), None)
        if ident is None:
            skipped["unmatched_name"] += 1
            continue
        if ident.player_id in seen:
            continue
        team = normalize_team_abbreviation(ident.team_abbreviation)
        placed = None
        for event, start, fetched in events.get(player, []):
            sides = _event_teams(event)
            if team in sides and len(sides) == 2:
                day = _tip_date(start) or _tip_date(fetched)
                if day is not None:
                    placed = (day, next(iter(sides - {team})))
                    break
        if placed is None:
            games = list(team_games.get(team, {}))
            if len(games) == 1:
                placed = games[0]
            elif len(games) > 1:
                skipped["ambiguous_game"] += 1
                continue
            else:
                skipped["no_game"] += 1
                continue
        day, opponent = placed
        opp_team_id = team_ids.get(opponent)
        if opp_team_id is None:
            skipped["no_opponent_id"] += 1
            continue
        seen.add(ident.player_id)
        pair = tuple(sorted((team, opponent)))
        rows.append({
            "player_id": ident.player_id,
            "player_name": ident.player_name,
            "team_abbreviation": team,
            "team_id": ident.team_id,
            "opp_team_id": opp_team_id,
            "game_id": f"{day.date()}|{pair[0]}|{pair[1]}",
            "game_date": day,
            "season_year": _season_label(day),
        })
    print(
        "slate",
        f"players={len(rows)}",
        " ".join(f"{name}={count}" for name, count in skipped.items()),
    )
    if not rows:
        raise ValueError("no quoted player matched a silver team and an odds event")
    return pd.DataFrame(rows)


history = _load_history()
tonight = build_tonight(history, quotes)
cutoff = tonight["game_date"].min()
before = history["game_date"] < cutoff
print(
    f"history rows before {cutoff.date()}: {int(before.sum()):,}",
    f"dropped {int((~before).sum()):,}",
)
history = history.loc[before].copy()
minutes_live = build_minutes_live(history, tonight, lineups=lineups)
rate_live = build_ppm_live(history, tonight, lineups=lineups)
print(
    f"minutes rows {len(minutes_live):,}",
    f"rate rows {len(rate_live):,}",
    f"starting=1 {int((minutes_live['starting'] == 1).sum())}",
    f"spread missing {int(rate_live['team_spread_canonical'].isna().sum())}",
)

card = price_slate(minutes_live, rate_live, quotes)
today = pd.Timestamp.now(tz="America/Los_Angeles").normalize().tz_localize(None)
tip_day = (
    pd.to_datetime(card["start_time"], utc=True, errors="coerce")
    .dt.tz_convert("America/Los_Angeles")
    .dt.normalize()
    .dt.tz_localize(None)
)
today_card = card.loc[tip_day.eq(today) & card["edge"].notna()].sort_values(
    "edge", ascending=False
)
print(
    f"today {today.date()} pacific",
    f"games {today_card['event_name'].nunique()}",
    f"lines {len(today_card)}",
)
today_card.head(20)


Loaded /Users/alexgonzalez/Documents/nba_quant/models/saved_models/min_nba_model_2026-04-12.joblib
Loaded /Users/alexgonzalez/Documents/nba_quant/models/saved_models/pts_nba_model_2026-04-12.joblib
minutes features: starting, starter_rank, start_rate_10, season_start_rate, starter_min_missing_teammates, total_stars_active, min_avg_lN, min_lag_1, min_ewm_hl_10, player_days_since_appearance, player_appearances_prev_7d, min_p80_10, min_p80_minus_p20_10
rate features: pts_per_min_ewm_hl_10, player_fga_share_10, ppm_trim_mean_20, current_team_pts_per_min, usg_wmean_10, season_pts_per_min, starting, fta_per_min_10, fga_per_min_10, fg3a_per_min_10, ppm_unstable, opp_def_rating_mean_10, min_mean_10
slate players=87 unmatched_name=1 no_game=15 ambiguous_game=0 no_opponent_id=0
history rows before 2026-10-24: 177,060 dropped 0
minutes rows 87 rate rows 87 starting=1 58 spread missing 87
today 2026-10-23 pacific games 6 lines 127


,book,player,prop,name_key,line,over_american,under_american,fetched_at,event_name,start_time,book_over,model_over,model_mean,model_median,edge,ev
0,fanduel,Dejounte Murray,points,dejounte murray,11.5,-112,-118.0,2026-10-24T02:25:39Z,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:43:00.000Z,0.493931,0.8720,19.0890,18.0,0.378069,0.650571
99,novig,Dejounte Murray,points,dejounte murray,14.5,163.0,-213.0,2026-10-23T19:25:37-07:00,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:35:00.000Z,0.358456,0.7260,19.0890,18.0,0.367544,0.909380
89,novig,Ace Bailey,points,ace bailey,9.5,-113.0,-127.0,2026-10-23T19:25:37-07:00,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:35:00.000Z,0.486718,0.8250,15.8620,15.0,0.338282,0.555088
12,fanduel,Ace Bailey,points,ace bailey,9.5,-112,-118.0,2026-10-24T02:25:39Z,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:43:00.000Z,0.493931,0.8250,15.8620,15.0,0.331069,0.561607
47,fanduel,GG Jackson II,points,gg jackson ii,10.5,106,-140.0,2026-10-24T02:25:39Z,Memphis Grizzlies @ Golden State Warriors,2026-10-24T02:14:00.000Z,0.454201,0.7305,14.5685,14.0,0.276299,0.504830
10,fanduel,Bennedict Mathurin,points,bennedict mathurin,5.5,-128,-104.0,2026-10-24T02:25:39Z,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:43:00.000Z,0.524085,0.7955,10.6440,10.0,0.271415,0.416984
93,novig,Keyonte George,points,keyonte george,14.5,-239.0,-182.0,2026-10-23T19:25:37-07:00,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:35:00.000Z,0.522077,0.7685,19.9860,20.0,0.246423,0.090048
7,fanduel,Keyonte George,points,keyonte george,15.5,102,-136.0,2026-10-24T02:25:39Z,New Orleans Pelicans @ Utah Jazz,2026-10-24T01:43:00.000Z,0.462093,0.7065,19.9860,20.0,0.244407,0.427130
42,fanduel,Damian Lillard,points,damian lillard,17.5,-106,-125.0,2026-10-24T02:25:39Z,Portland Trail Blazers @ Sacramento Kings,2026-10-24T02:11:00.000Z,0.480847,0.7195,21.8365,22.0,0.238653,0.398274
136,novig,Damian Lillard,points,damian lillard,19.5,138.0,-194.0,2026-10-23T19:25:37-07:00,Portland Trail Blazers @ Sacramento Kings,2026-10-24T02:05:00.000Z,0.389033,0.6170,21.8365,22.0,0.227967,0.468460


In [22]:
card = price_slate(minutes_live, rate_live, quotes)
today = pd.Timestamp.now(tz="America/Los_Angeles").normalize().tz_localize(None)
tip_day = (
    pd.to_datetime(card["start_time"], utc=True, errors="coerce")
    .dt.tz_convert("America/Los_Angeles")
    .dt.normalize()
    .dt.tz_localize(None)
)
today_card = card.loc[tip_day.eq(today) & card["edge"].notna()].sort_values(
    "edge", ascending=False
)
print(
    f"today {today.date()} pacific",
    f"games {today_card['event_name'].nunique()}",
    f"lines {len(today_card)}",
)
today_card

today 2026-10-22 pacific games 2 lines 39


,book,player,prop,name_key,line,over_american,under_american,fetched_at,event_name,start_time,book_over,model_over,model_mean,model_median,edge,ev
238,novig,Jaylen Brown,points,jaylen brown,19.5,133.0,-3126.0,2026-10-22T18:41:24-07:00,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:05:00.000Z,0.306958,0.6705,22.8700,23.0,0.363542,0.562265
3,fanduel,VJ Edgecombe,points,vj edgecombe,11.5,220.0,-310.0,2026-10-23T01:42:37Z,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:14:00.000Z,0.292439,0.6490,14.8525,14.0,0.356561,1.076800
5,fanduel,James Harden,points,james harden,14.5,124.0,-166.0,2026-10-23T01:42:37Z,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:14:00.000Z,0.417032,0.7355,19.2970,19.0,0.318468,0.647520
4,fanduel,Jaylen Brown,points,jaylen brown,17.5,108.0,-144.0,2026-10-23T01:42:37Z,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:14:00.000Z,0.448926,0.7540,22.8700,23.0,0.305074,0.568320
6,fanduel,Jarrett Allen,points,jarrett allen,14.5,194.0,-270.0,2026-10-23T01:42:37Z,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:14:00.000Z,0.317924,0.5230,15.5205,15.0,0.205076,0.537620
1,fanduel,Peyton Watson,points,peyton watson,8.5,210.0,-300.0,2026-10-23T01:42:37Z,Cleveland Cavaliers @ Philadelphia 76ers,2026-10-22T23:14:00.000Z,0.300752,0.4855,9.3270,8.0,0.184748,0.505050
20,fanduel,Christian Braun,points,christian braun,10.5,-111.0,-115.0,2026-10-23T01:42:37Z,Denver Nuggets @ Oklahoma City Thunder,2026-10-23T01:50:00.000Z,0.495845,0.5315,11.9160,11.0,0.035655,0.010329
246,novig,Jamal Murray,points,jamal murray,21.5,-102.0,-122.0,2026-10-22T18:41:24-07:00,Denver Nuggets @ Oklahoma City Thunder,2026-10-23T01:35:00.000Z,0.478853,0.4765,21.3695,21.0,-0.002353,-0.056343
14,fanduel,Chet Holmgren,points,chet holmgren,15.5,100.0,-128.0,2026-10-23T01:42:37Z,Denver Nuggets @ Oklahoma City Thunder,2026-10-23T01:50:00.000Z,0.471074,0.4525,15.2805,15.0,-0.018574,-0.095000
19,fanduel,Jamal Murray,points,jamal murray,21.5,-111.0,-115.0,2026-10-23T01:42:37Z,Denver Nuggets @ Oklahoma City Thunder,2026-10-23T01:50:00.000Z,0.495845,0.4765,21.3695,21.0,-0.019345,-0.094221
